[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pythonanywhere/pypath/blob/main/notebooks/module6/07-cv-applications.ipynb)

# Module 6.7 — End-to-End CV Application
**Module 6: Computer Vision** | Estimated time: 40 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Build a complete image classification pipeline from data to deployment
- Prepare a dataset using `ImageFolder` and `DataLoader`
- Fine-tune ResNet18 with a full training + validation loop
- Plot a confusion matrix to evaluate model performance
- Save and reload a trained model with `torch.save` / `torch.load`
- Write a FastAPI inference endpoint and a Gradio demo interface

In [ ]:
!pip install gradio --quiet

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import os, time, json
from pathlib import Path

# Optional: sklearn for confusion matrix
try:
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# We use CIFAR-10 as stand-in for any 10-class image dataset
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
NUM_CLASSES = len(CLASSES)
torch.manual_seed(0)

## Step 1 — Dataset Preparation

In a real project you would organise images in the `ImageFolder` format:
```
dataset/
    train/
        class_a/  img1.jpg  img2.jpg ...
        class_b/  img1.jpg ...
    val/
        class_a/  ...
        class_b/  ...
```

Here we use CIFAR-10 (available directly from `torchvision`) and mimic the same pipeline you would use with your own `ImageFolder` dataset.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_tfm = transforms.Compose([
    transforms.Resize(64),                         # upscale 32→64 for ResNet
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(64, padding=8),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

val_tfm = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

print('Downloading CIFAR-10...')
train_ds = torchvision.datasets.CIFAR10('/tmp/cifar10', train=True,
                                        download=True, transform=train_tfm)
val_ds   = torchvision.datasets.CIFAR10('/tmp/cifar10', train=False,
                                        download=True, transform=val_tfm)

# Use a manageable subset for demo
TRAIN_N, VAL_N = 8000, 1600
train_ds = Subset(train_ds, range(TRAIN_N))
val_ds   = Subset(val_ds,   range(VAL_N))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)}  ({TRAIN_N} samples)')
print(f'Val   batches: {len(val_loader)}   ({VAL_N} samples)')

# Show class distribution
all_labels = [train_ds.dataset.targets[i] for i in train_ds.indices]
counts = np.bincount(all_labels)
plt.figure(figsize=(10, 3))
plt.bar(CLASSES, counts)
plt.title('Training Class Distribution'); plt.xlabel('Class'); plt.ylabel('Count')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## Step 2 — Model: ResNet18 Transfer Learning

We partially unfreeze the model to get better accuracy:
- Freeze layers 1-3 of ResNet18 (keep low-level features fixed)
- Fine-tune layer 4 and the new FC head (learn task-specific features)

In [ ]:
def build_model(num_classes, unfreeze_layer4=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze all parameters first
    for param in model.parameters():
        param.requires_grad = False

    # Optionally unfreeze layer4 (the deepest residual block)
    if unfreeze_layer4:
        for param in model.layer4.parameters():
            param.requires_grad = True

    # Replace classification head
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Linear(256, num_classes)
    )
    return model

model = build_model(NUM_CLASSES).to(DEVICE)

frozen  = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Frozen params   : {frozen:,}')
print(f'Trainable params: {trained:,}')

# Two-group optimizer: lower LR for backbone, higher for head
optimizer = optim.AdamW([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 5e-4},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

## Step 3 — Training and Validation Loop

In [ ]:
def run_epoch(model, loader, optimizer, criterion, training):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc']}
NEPOCHS = 8

print(f'Training for {NEPOCHS} epochs on {DEVICE}...\n')
best_val_acc = 0.0

for epoch in range(1, NEPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(model, train_loader, optimizer, criterion, True)
    vl_loss, vl_acc = run_epoch(model, val_loader,   optimizer, criterion, False)
    scheduler.step()

    for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],
                    [tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), '/tmp/best_model.pt')
        tag = ' ← best'
    else:
        tag = ''

    print(f'Epoch {epoch:2d}/{NEPOCHS}  '
          f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f}  '
          f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.3f}  '
          f'{time.time()-t0:.1f}s{tag}')

print(f'\nBest validation accuracy: {best_val_acc*100:.1f}%')

## Step 4 — Plotting Training Curves and Confusion Matrix

In [ ]:
epochs = range(1, NEPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train')
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)

plt.suptitle('Training Curves — ResNet18 Fine-tuned on CIFAR-10', fontweight='bold')
plt.tight_layout(); plt.show()

# Confusion matrix
model.load_state_dict(torch.load('/tmp/best_model.pt', map_location=DEVICE))
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

if HAS_SKLEARN:
    cm = confusion_matrix(all_true, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES)
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.title('Confusion Matrix (Validation Set)')
    plt.tight_layout(); plt.show()
else:
    # Simple matplotlib fallback
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, p in zip(all_true, all_preds):
        cm[t, p] += 1
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, cmap='Blues')
    plt.xticks(range(NUM_CLASSES), CLASSES, rotation=45, ha='right')
    plt.yticks(range(NUM_CLASSES), CLASSES)
    plt.colorbar()
    plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

acc = np.trace(cm) / np.sum(cm)
print(f'Overall accuracy: {acc*100:.1f}%')

## Step 5 — Save and Reload Model

There are two common ways to save a PyTorch model:
1. **State dict only** (recommended): saves only the weights; you must recreate the architecture first.
2. **Full model**: saves both architecture and weights using `pickle` — less portable across PyTorch versions.

In [ ]:
# Save state dict + metadata together
checkpoint = {
    'epoch':       NEPOCHS,
    'val_acc':     best_val_acc,
    'classes':     CLASSES,
    'state_dict':  model.state_dict(),
    'optimizer':   optimizer.state_dict(),
}
torch.save(checkpoint, '/tmp/classifier_checkpoint.pt')
print(f'Checkpoint saved: {os.path.getsize("/tmp/classifier_checkpoint.pt")//1024} KB')

# Reload and verify
def load_model(path, num_classes):
    ckpt  = torch.load(path, map_location=DEVICE)
    m     = build_model(num_classes).to(DEVICE)
    m.load_state_dict(ckpt['state_dict'])
    m.eval()
    return m, ckpt['classes'], ckpt['val_acc']

loaded_model, loaded_classes, saved_acc = load_model('/tmp/classifier_checkpoint.pt', NUM_CLASSES)
print(f'Loaded model — saved val acc: {saved_acc*100:.1f}%')
print(f'Classes: {loaded_classes}')

# Quick inference test
import torchvision.transforms.functional as TF
from PIL import Image as PILImage

def predict(model, pil_image, classes):
    tfm = transforms.Compose([
        transforms.Resize(64),
        transforms.CenterCrop(64),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    tensor = tfm(pil_image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
    top5   = probs.topk(5)
    return [(classes[idx], prob.item()) for idx, prob in
            zip(top5.indices, top5.values)]

# Test on the first validation image
raw_val = torchvision.datasets.CIFAR10('/tmp/cifar10', train=False, download=False)
pil_img, true_lbl = raw_val[0]
predictions = predict(loaded_model, pil_img, CLASSES)
print(f'\nTrue label: {CLASSES[true_lbl]}')
print('Top-5 predictions:')
for cls, prob in predictions:
    bar = '█' * int(prob * 30)
    print(f'  {cls:12s}  {prob*100:5.1f}%  {bar}')

## Step 6 — FastAPI Inference Endpoint

In [ ]:
%%writefile /tmp/app.py
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from PIL import Image
import io

DEVICE = 'cpu'
CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

app = FastAPI(title='Image Classifier API')

# Build model
def build_model():
    m = models.resnet18(weights=None)
    m.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(m.fc.in_features, 256),
        nn.ReLU(),
        nn.Linear(256, len(CLASSES))
    )
    return m

model = build_model().to(DEVICE)
try:
    ckpt = torch.load('/tmp/classifier_checkpoint.pt', map_location=DEVICE)
    model.load_state_dict(ckpt['state_dict'])
except FileNotFoundError:
    pass   # weights not loaded — demo only
model.eval()

transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

@app.get('/')
def root():
    return {'message': 'Image Classifier API', 'classes': CLASSES}

@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    contents = await file.read()
    img = Image.open(io.BytesIO(contents)).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs  = torch.softmax(logits, dim=1)[0]
    top5 = probs.topk(5)
    return JSONResponse({
        'predictions': [
            {'class': CLASSES[i], 'confidence': round(p, 4)}
            for i, p in zip(top5.indices.tolist(), top5.values.tolist())
        ]
    })

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='0.0.0.0', port=8000)
print('FastAPI app written to /tmp/app.py')
print('To run: !python /tmp/app.py &')
print('To test: !curl -X POST http://localhost:8000/predict -F "file=@image.jpg"')

## Step 7 — Gradio Demo Interface

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image as PILImage

def classify_image(pil_image):
    """Gradio-compatible inference function."""
    if pil_image is None:
        return {}
    preds = predict(loaded_model, pil_image, CLASSES)
    return {cls: float(f'{prob:.4f}') for cls, prob in preds}

# Build Gradio interface
demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type='pil', label='Upload an image'),
    outputs=gr.Label(num_top_classes=5, label='Predictions'),
    title='CIFAR-10 Image Classifier',
    description=(
        'Upload any image. The model was fine-tuned on CIFAR-10 (10 classes).\n'
        'Best results with small 32×32 style images of the 10 CIFAR categories.'
    ),
    examples=[],
    theme=gr.themes.Soft(),
)

print('Gradio demo created.')
print('To launch in Colab: demo.launch(share=True)')
print('\nTo launch now (no share link):')
print('  demo.launch()')

# Uncomment to launch:
# demo.launch(share=True)

## Summary — End-to-End Pipeline

```
Raw Data
    ↓  ImageFolder + transforms (augmentation)
DataLoader (batched, shuffled)
    ↓  ResNet18 (frozen backbone + new head)
Training loop (AdamW + CosineAnnealing)
    ↓  Save best checkpoint
Evaluation (confusion matrix, per-class accuracy)
    ↓  torch.save / torch.load
Deployment (FastAPI REST endpoint + Gradio UI)
```

## Practice Exercises

**Exercise 1 — Per-Class Accuracy:**  
From the confusion matrix, compute the per-class precision and recall. Which class has the lowest recall? Look at misclassified examples from that class — can you understand why the model struggles?

**Exercise 2 — Grad-CAM Visualisation:**  
Install `pip install grad-cam` and use `pytorch_grad_cam` to visualise which parts of the image the model focuses on when making a prediction. Run it on 5 correctly classified and 5 misclassified examples.

**Exercise 3 — Real Dataset:**  
Download the Oxford 102 Flowers dataset (`torchvision.datasets.Flowers102`). Replace CIFAR-10 in this pipeline with Flowers102 (102 classes). Adjust the model output size and retrain. What accuracy do you achieve after 10 epochs of fine-tuning?